In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from scipy import stats

Таким образом можно очистить все таблицы

In [ ]:
res = requests.delete("http://127.0.0.1:8001/api/v1/users")
res = requests.delete("http://127.0.0.1:8001/api/v1/things")
res = requests.delete("http://127.0.0.1:8001/api/v1/sales")

Можно добавлять пользователей/товары/ивенты продаж

In [44]:
params = {
    "user_name": "Reinhard",
    "user_age": 23,
    "bought_premium": True
}
res = requests.post("http://127.0.0.1:8001/api/v1/users", json=params)
res.json()

{'user_id': 667,
 'user_name': 'Reinhard',
 'user_age': 23,
 'bought_premium': True}

Можно сгенерировать чтобы не добавлять руками

In [35]:
requests.post("http://127.0.0.1:8001/api/v1/users/generate_users", params = {"count": 666})
requests.post("http://127.0.0.1:8001/api/v1/things/generate_things", params = {"count": 322})
requests.post("http://127.0.0.1:8001/api/v1/sales/generate_sales", params = {"count": 1336})

<Response [200]>

Получим сводки по пользователям, товарам, продажам

In [19]:
res = requests.get("http://127.0.0.1:8001/api/v1/users/summary")
res.json()

{'cnt_users': 666,
 'cnt_premium_users': 344.0,
 'frac_premium_users': 0.5165165165165165,
 'quartiles(age)': {'0.25': 22.0, '0.5': 48.0, '0.75': 73.75}}

In [20]:
res = requests.get("http://127.0.0.1:8001/api/v1/things/summary")
res.json()

{'cnt_things': 322,
 'clothes': {'price_quartiles': {'0.25': 3064.75,
   '0.5': 3868.5,
   '0.75': 4229.5},
  'cnt_things': 66,
  'frac_things': 0.20496894409937888},
 'electronics': {'price_quartiles': {'0.25': 5327.75,
   '0.5': 6124.0,
   '0.75': 6606.75},
  'cnt_things': 58,
  'frac_things': 0.18012422360248448},
 'food': {'price_quartiles': {'0.25': 531.25, '0.5': 1109.5, '0.75': 1882.75},
  'cnt_things': 70,
  'frac_things': 0.21739130434782608},
 'toys': {'price_quartiles': {'0.25': 5387.25, '0.5': 6243.0, '0.75': 6936.25},
  'cnt_things': 62,
  'frac_things': 0.19254658385093168},
 'weapons': {'price_quartiles': {'0.25': 7110.5,
   '0.5': 7853.5,
   '0.75': 8530.25},
  'cnt_things': 66,
  'frac_things': 0.20496894409937888}}

In [5]:
res = requests.get("http://127.0.0.1:8001/api/v1/sales/summary")
res.json()

{'cnt_sales': 1337,
 'avg_sales_per_user': 2.3292682926829267,
 'most_active_users': [{'user_id': 4013, 'cnt_sales': 7},
  {'user_id': 3740, 'cnt_sales': 6},
  {'user_id': 3749, 'cnt_sales': 6},
  {'user_id': 3559, 'cnt_sales': 6},
  {'user_id': 3986, 'cnt_sales': 6}],
 'most_popular_things': [{'thing_id': 2561, 'cnt_sales': 15},
  {'thing_id': 2537, 'cnt_sales': 14},
  {'thing_id': 2607, 'cnt_sales': 12},
  {'thing_id': 2500, 'cnt_sales': 11},
  {'thing_id': 2649, 'cnt_sales': 11}],
 'cnt_per_category': {'clothes': 460,
  'food': 354,
  'weapons': 239,
  'electronics': 171,
  'toys': 112,
  'null': 1},
 'frac_per_category': {'clothes': 0.34405385190725507,
  'food': 0.2647718773373224,
  'weapons': 0.1787584143605086,
  'electronics': 0.12789827973074047,
  'toys': 0.08376963350785341,
  'null': 0.0007479431563201197},
 'cnt_payment_type': {'nalik': 739, 'card': 598},
 'frac_payment_type': {'nalik': 0.5527299925205684,
  'card': 0.4472700074794316}}

Проведем кластеризацию, получив в ответе центроиды

In [39]:
res = requests.post("http://127.0.0.1:8001/api/v1/users/cluster_users",params={"n_clusters":4})
res.json()

[{'cluster_size': 185,
  'cluster': 0,
  'cnt_sales': 2,
  'avg_price': 3358.8046589446594,
  'med_price': 3414.9621621621623,
  'user_age': 60.13513513513514,
  'bought_premium': False,
  'mode_category': 'clothes'},
 {'cluster_size': 193,
  'cluster': 1,
  'cnt_sales': 2,
  'avg_price': 4998.236787564767,
  'med_price': 5066.577720207254,
  'user_age': 48.398963730569946,
  'bought_premium': True,
  'mode_category': 'clothes'},
 {'cluster_size': 105,
  'cluster': 2,
  'cnt_sales': 1,
  'avg_price': 1113.228185941043,
  'med_price': 929.8571428571429,
  'user_age': 55.733333333333334,
  'bought_premium': False,
  'mode_category': 'weapons'},
 {'cluster_size': 103,
  'cluster': 3,
  'cnt_sales': 1,
  'avg_price': 7372.600647249191,
  'med_price': 7507.665048543689,
  'user_age': 33.116504854368934,
  'bought_premium': False,
  'mode_category': 'electronics'}]

Выбираем всех пользователей из какого-то кластера

In [43]:
res = requests.get("http://127.0.0.1:8001/api/v1/users/clusters/3")
res.json()

{'cluster_size': 103,
 'users': [{'user_id': 3,
   'cluster': 3,
   'cnt_sales': 2,
   'avg_price': 7595,
   'med_price': 7595,
   'user_age': 18,
   'bought_premium': True,
   'mode_category': 'electronics'},
  {'user_id': 4,
   'cluster': 3,
   'cnt_sales': 1,
   'avg_price': 6673,
   'med_price': 6673,
   'user_age': 12,
   'bought_premium': True,
   'mode_category': 'electronics'},
  {'user_id': 7,
   'cluster': 3,
   'cnt_sales': 1,
   'avg_price': 7699,
   'med_price': 7699,
   'user_age': 18,
   'bought_premium': False,
   'mode_category': 'weapons'},
  {'user_id': 10,
   'cluster': 3,
   'cnt_sales': 2,
   'avg_price': 7752,
   'med_price': 7752,
   'user_age': 1,
   'bought_premium': True,
   'mode_category': 'weapons'},
  {'user_id': 13,
   'cluster': 3,
   'cnt_sales': 3,
   'avg_price': 6483.66666666667,
   'med_price': 6389,
   'user_age': 7,
   'bought_premium': True,
   'mode_category': 'electronics'},
  {'user_id': 36,
   'cluster': 3,
   'cnt_sales': 1,
   'avg_price':

Теперь проведём парочку тестов предположив что имеем дело с распределением Бернулли и всё что должно быть независимым с чем-либо является таковым.

In [6]:
# big sample test for difference in means
# only for bernoulli
def bstest(x, y, alpha=0.05):

    nx = len(x)
    ny = len(y)
    xm = x.mean()
    ym = y.mean()
    diff = xm - ym

    p_est = (sum(x) + sum(y)) / (nx + ny)
    var_pooled = p_est * (1 - p_est)

    z_stat =  diff / ( np.sqrt(var_pooled) * np.sqrt(1 / nx + 1 / ny) )
    if z_stat >= 0:
        pval = 2 * (1 - stats.norm.cdf(z_stat))
    else:
        pval = 2 * stats.norm.cdf(z_stat)

    
    q = (-1) * stats.norm.ppf(alpha / 2)
    half_len = q * np.sqrt(xm * (1 - xm) / nx + ym * (1 - ym) / ny)
    l = diff - half_len
    r = diff + half_len
    
    return pval, (l, r)

### Test card/nalik proportions in groups by age


In [7]:
res = requests.get("http://127.0.0.1:8001/api/v1/users")
res = res.json()
users_df = pd.DataFrame(res, columns=res[0].keys())

In [8]:
res = requests.get("http://127.0.0.1:8001/api/v1/sales")
res = res.json()
sales_df = pd.DataFrame(res, columns=res[0].keys())

In [9]:
df = sales_df.set_index('user_id').join(users_df.set_index('user_id'))
df.head()

,sale_id,thing_id,count,payment_type,user_name,user_age,bought_premium
user_id,,,,,,,
5,6006,1,23,card,NaN,NaN,NaN
3519,6007,2465,6,nalik,yui lando,36.0,True
3619,6008,2422,7,nalik,yui lando,25.0,False
3724,6009,2582,6,nalik,alex lando,27.0,True
3459,6010,2392,3,card,makise le monon,32.0,True


In [10]:
df["indicator"] = df["payment_type"].map({"nalik":1, "card":0}).to_numpy()

In [11]:
x = df[df["user_age"] <= 18]["indicator"]
y = df[df["user_age"] > 18]["indicator"]

In [12]:
print(x.mean(), y.mean())

0.34156378600823045 0.6001829826166514


In [13]:
pval, (l, r) = bstest(x,y)

Получается, что различие в вероятностях использования наличных при оплате покупки
внутри разных возрастных групп является статистически значимым при любом разумном уровне значимости.

In [14]:
print(f"p-value = {pval}")
print(f"95% conf interval for diff in props: ({l:.3f},{r:.3f})")

p-value = 2.2262633846294162e-13
95% conf interval for diff in props: (-0.325,-0.192)


### Test for premium proportions in group by age


In [15]:
x = users_df[users_df["user_age"] <= 18]["bought_premium"].astype(int)
y = users_df[users_df["user_age"] > 18]["bought_premium"].astype(int)

In [16]:
print(x.mean(), y.mean())

0.5076923076923077 0.5186567164179104


In [17]:
pval, (l, r) = bstest(x,y)

А вот различие в вероятностях приобретения премиума уже не является статистически значимым.

In [18]:
print(f"p-value = {pval}")
print(f"95% conf interval for diff in props: ({l:.3f},{r:.3f})")

p-value = 0.822427419206095
95% conf interval for diff in props: (-0.107,0.085)
